# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant Schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Identifier: {meta.identifier}")
print(f"Published date: {meta.datePublished}")
print(f"Version: {meta.version}")

## 2. Data Overview
Review available record sets, fields (columns), and their IDs.

We will list record set `@id`s, and for each, show a sample of records and their field `@id`s.

In [ ]:
# List available record sets by @id and display column and field metadata per set
record_sets = dataset.record_sets
print("Available Record Sets (by @id):\n")
for rs in record_sets:
    print(f"- {rs['@id']}")

# Show columns for each record set
fields_by_rs = {}
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        fields_by_rs[rs['@id']] = [f['@id'] for f in fields]
        print("  Field @ids:")
        for f in fields:
            print(f"    - {f['@id']}")
    else:
        fields_by_rs[rs['@id']] = []
        print("  (No explicit fields defined)")

# Optionally preview a few records from the first record set
if record_sets:
    first_rs_id = record_sets[0]['@id']
    print(f"\nSample records in record set: {first_rs_id}")
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        if i>=2:
            break
        print(rec)
else:
    print('No record sets found in the dataset.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We will use the record set and field `@id`s found in the overview above.

This dataset typically has one main record set; we'll extract all as DataFrames by their `@id`.

In [ ]:
# Extract data into pandas DataFrames for all record sets
all_rs_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for recset_id in all_rs_ids:
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for record set {recset_id}")

# Display columns of the main record set DataFrame
main_rs_id = all_rs_ids[0] if all_rs_ids else None
if main_rs_id and not dataframes[main_rs_id].empty:
    print(f"\nColumns in record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print('No data available to extract.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. This prepares the dataset for further analysis.

We first identify a numeric field among the columns for demonstration. We will filter and normalize it and, if possible, group by a categorical column.

In [ ]:
# Choose the main DataFrame and analyze its numeric field(s)
import numpy as np

df = dataframes[main_rs_id]

print(f"Columns: {df.columns.tolist()}")

# Attempt to find one numeric field (int or float), else use a likely candidate
possible_numeric_fields = [col for col in df.columns if df[col].dtype in [np.int64, np.float64]]
if not possible_numeric_fields:
    # Guess a field, e.g. columns with 'Age', 'Interval', or similar
    possible_numeric_fields = [col for col in df.columns if any(x in col.lower() for x in ['age', 'interval', 'duration', 'year'])]

if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
    print(f"Using numeric field for EDA: {numeric_field_id}")
    # Try converting if necessary
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
else:
    print("No obvious numeric field found.")

# EDA: Filter, normalize, group (if group field exists)
if possible_numeric_fields:
    threshold = df[numeric_field_id].dropna().mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Find group/categorical column (e.g. 'Sex', 'AnatomicalSite', etc)
    possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
    group_field_id = possible_group_fields[0] if possible_group_fields else None

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No grouping (categorical) field found for this record set.")
else:
    print("No numeric fields to analyze in this record set.")

## 5. Visualization
Visualize distributions or relationships from the dataset. Here we plot the numeric field's distribution and, if possible, group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if possible_numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have loaded the FAIR² colorectal cancer survivors dataset using `mlcroissant`, explored its structure, extracted key record sets and performed initial exploratory analysis on available numeric and grouping variables.

This workflow demonstrates:
- How to utilize Croissant `@id` references for all queries and field access
- Basic steps for wrangling and visualizing Croissant-conformant datasets

You can adapt and extend this notebook for more detailed analyses or modeling tasks tailored to your research questions.